In [ ]:
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import snapshot_download, login
from datasets import load_dataset



In [ ]:
from peft import PeftModel, PeftConfig
import json
import re
from IPython.display import Markdown, display
import numpy as np
import json
from tqdm import tqdm
from torch.nn.attention import SDPBackend, sdpa_kernel

In [ ]:
torch.backends.cuda.flash_sdp_enabled()

In [ ]:
BASE_MODEL_DIR = "/data/Iman/shared_LLM_models/Qwen2.5-Math-7B/"
ADAPTER_DIR = "./finetuned_models/lora-qwen-atc-finetuned-lora-adapters/"

## 1. Load tokenizer and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# attach LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR)

model = model.merge_and_unload()
# model.eval()

In [ ]:
# saving the merged model
merged_model_dir = "./finetuned_models/merged-qwen-atc-model/"

# model.save_pretrained(merged_model_dir)
# tokenizer.save_pretrained(merged_model_dir)

In [ ]:
# load the merged model
quantization_config = BitsAndBytesConfig(
    load_in_16bit=True,
)
# model = AutoModelForCausalLM.from_pretrained(
#     merged_model_dir,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     # quantization_config=quantization_config,
#     # attn_implementation="sdpa",
#     # trust_remote_code=True,
# )
model = AutoModelForCausalLM.from_pretrained(
    merged_model_dir, # BASE_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    # quantization_config=quantization_config,
    # attn_implementation="sdpa",
    # trust_remote_code=True,
)

model.eval()


## 2. Extracting Data and formating

In [ ]:
# read the data from json file
file_path = 'ownship_prompts_data.jsonl'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

# ds = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.2, seed=42)
# test = ds['test']

def format_prompt(example):
    return (f"### System:\n{example['system_prompt'].strip()}\n\n"
            f"### User:\n{example['user_prompt'].strip()}\n\n"
            f"### Assistant:\n{example['prompt_response'].strip()}"
    )

In [ ]:
data[0]  # Display the first two entries to verify the data structure

In [ ]:
ex_prompt = format_prompt(data[100])
print(ex_prompt)  # Format the first entry to verify the prompt structure

In [ ]:
inputs = tokenizer(ex_prompt, return_tensors="pt").to("cuda")
inputs

In [ ]:
with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    output_tokens = base_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.0,
        top_p=0.95,
        # repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

response_text = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
# print("Full Response:")
response_text = "".join(response_text)
# print(response_text)
assistant_output = response_text.split("### Assistant:")[-1].strip()
print("Assistant's Response:")
print(assistant_output)

In [ ]:
# locate the word "Accelerate|Hold|Decelerate" in the response
match = re.search(r'\b(Accelerate|Hold|Decelerate)\b', assistant_output, re.IGNORECASE)
if match:
    predicted_action = match.group(1).capitalize()
    print(f"Predicted Action: {predicted_action}")

## 3. Accuracy or batch evaluation

In [ ]:
# test data = randomly sample 1000 entries from data
import random
test_data = random.sample(data, 1000)
test_data

In [ ]:
def retrieve_action(response):
    match = re.search(r'\b(Accelerate|Hold|Decelerate)\b', response, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return None

In [ ]:
test_data

In [ ]:
def generate_response(system_prompt, user_prompt):
    prompt = f"### System:\n{system_prompt.strip()}\n\n### User:\n{user_prompt.strip()}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
    decoded = "".join(decoded)
    return decoded.split("### Assistant:")[-1].strip()



correct, total = 0, 0
for ex in tqdm(test_data):
    pred_response = generate_response(ex['system_prompt'], ex['user_prompt'])
    # the last word of the response should match "Acelerate|Hold|Decelerate"
    # so we can search for that word in the response and compare

    if pred_response.lower() == ex['prompt_response'].strip().lower():
        correct += 1
    total += 1

accuracy = correct / total
print(f"Accuracy over {total} samples: {accuracy:.4f}")

In [ ]:
def build_prompt(example):
    return (f"### System:\n{example['system_prompt'].strip()}\n\n"
            f"### User:\n{example['user_prompt'].strip()}\n\n"
            f"### Assistant:\n"
    )

def generate_batch_responses(examples):
    responses = []
    prompts = [build_prompt(example) for example in examples]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            # top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            # use_cache=True,
        )

    decoded = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
    # decoded = "".join(decoded)
    # assistant_response = decoded.split("### Assistant:")[-1].strip()
    # responses.append(assistant_response)
    return decoded

decoded = generate_batch_responses(test_data[:4])
decoded
# for i, response in enumerate(decoded):
#     assistant_response = response.split("### Assistant:")[-1].strip()
#     print(f"Response {i+1}: {assistant_response}")

In [ ]:
import math

def compute_perplexity(model, tokenizer, dataset):
    losses = []
    for ex in dataset:
        prompt = f"### System:\n{ex['system_prompt']}\n\n### User:\n{ex['user_prompt']}\n\n### Assistant:\n{ex['prompt_response']}"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        losses.append(loss.item())
    return math.exp(np.mean(losses))

perplexity = compute_perplexity(model, tokenizer, test_data)
print(f"Perplexity on test data: {perplexity:.4f}")

## Batched evaluation (not working)

In [ ]:
data_loaded = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.005, seed=42)
train_data = data_loaded['train']
test_data = data_loaded['test']

In [ ]:
test_data

In [ ]:
test_data[0:10]

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 4
MAX_LEN = 1024


def collate_fn(examples):
    prompts = [build_prompt(ex) for ex in examples]
    # print(prompts[0])
    refs = [ex['prompt_response'] for ex in examples]
    encodings = tokenizer(prompts, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding=True)
    encodings["prompts"] = prompts # keep raw text for decoding later
    encodings["refs"] = refs
    return encodings

dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
predictions, references = [], []

for batch in tqdm(dataloader, desc="Evaluating"):
    input_ids = batch['input_ids'].to("cuda")
    attention_mask = batch['attention_mask'].to("cuda")
    prompts = batch['prompts']
    refs = batch['refs']

    # print(batch)
    with torch.no_grad():
        output_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
    for text, ex in zip(decoded, prompts):
        assistant_response = text.split("### Assistant:")[-1].strip()
        predictions.append(assistant_response)

    references.extend(refs)

In [ ]:
len(predictions), len(references)

In [ ]:
predictions[0:5], references[0:5]

In [ ]:
def accuracy_score(preds, refs):
    correct = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    return correct / len(refs)

acc = accuracy_score(predictions, references)
print(f"Batched Evaluation Accuracy: {acc:.4f}")